Run all when generating a PDB, as the code overrides positions compiled, so it has to compile the initial positions again, otherwise you would be over twisting.

In [1]:
import MDAnalysis as md
import matplotlib.pyplot as plt
import numpy as np
import cyltransf as ct

filedir = "./"

fib = md.Universe(filedir + "colfib.pdb")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Twist is not dependant upon the radius, or the way we compute it, as the "translation" we are doing is upon the angle, then in a bigger radius theres already a similar scaling, due to the geometry in it.

The angle in a bigger radius, in the same $z$, makes a bigger geometrical translation. The arc is what is aumenting with the radius, and thats the geometrical thing.

In [ ]:
prot = fib.select_atoms("protein")
pos = prot.positions
z_middle = (max(pos[:,2]) + min(pos[:,2]))/2

# Center
com = prot.center_of_mass()

# Transformation function
m = -0.15 # Its in atomic units! (in nm is 1.5)
twist = lambda z: np.deg2rad(m)*(z - z_middle) # so that in the middle is the shifting point of translation, ie, twisted induced

rad, phi, z = ct.cyl_proj(pos, x_c=com[0], y_c=com[1])

phi_twisted = phi + twist(z)

x_tw, y_tw = rad*np.cos(phi_twisted) + com[0], rad*np.sin(phi_twisted) + com[1]

In [3]:
new_pos = np.column_stack([x_tw, y_tw, z]).astype(np.float64)

# Write back into the Universe (protein only)
prot.positions = new_pos

# Export whole system with protein twisted and everything else unchanged
fib.atoms.write("colfib-tw.pdb")
print("Wrote PDB")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn("Unit cell dimensions not found. "


Wrote PDB
